# 05 — Research Walkthrough: How Does GPT-2 Do Factual Recall?

This notebook is a **guided investigation** showing how the platform's experiment
families compose into a real mechanistic interpretability research workflow.

**Question:** When GPT-2 Small completes *"The capital of France is"* → *" Paris"*,
which components are responsible, and how do we verify it?

We answer this question in eight steps using six experiment families, moving from
correlation → causation → feature-level → automated follow-up.

---
**Prerequisite:** gpt2-small must be in your HuggingFace cache.  
If not: `huggingface-cli download gpt2` or set `HF_HOME` before running.

## Section 1: Set up

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add src to path when running from notebooks/ without package install.
repo_root = Path("__file__").parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# Verify TransformerLens is available (part of the 'interp' optional group).
try:
    import transformer_lens  # noqa: F401
except ImportError as exc:
    raise SystemExit(
        "TransformerLens not found. Install with: uv sync --extra interp"
    ) from exc

# Verify gpt2-small is cached.
from transformers import AutoConfig  # type: ignore[import-untyped]
try:
    AutoConfig.from_pretrained("gpt2", local_files_only=True)
except Exception:
    print("WARNING: gpt2 not found in HF cache.")
    print("Run: huggingface-cli download gpt2  (or set HF_HOME)")

from mech_interp.experiments.registry import ExperimentRegistry, load_experiment_spec
from mech_interp.storage.sqlite_store import SQLiteResultStore

# Ephemeral store for this walkthrough — nothing persists beyond the session.
import tempfile
tmpdir = Path(tempfile.mkdtemp(prefix="walkthrough_"))
store = SQLiteResultStore(
    database_path=tmpdir / "walkthrough.db",
    artifact_dir=tmpdir / "artifacts",
)
store.initialize()

# List all registered experiment families.
from mech_interp.experiments.families import ExperimentFamily
families = [f.value for f in ExperimentFamily]
print(f"Registered families ({len(families)}):")
for f in families:
    print(f"  {f}")

## Section 2: Direct logit attribution — where does the answer come from?

**Direct Logit Attribution (DLA)** decomposes the final logit as a sum of
per-component contributions *without any ablation* — one forward pass.

For each component `c` (attention head or MLP at each layer):

```
DLA(c) = output_of_c[last_pos] @ W_U @ (e_Paris - e_London)
```

Positive score → the component writes in the direction of *" Paris"*.

In [ ]:
import matplotlib.pyplot as plt

from mech_interp.experiments.direct_logit_attribution import DirectLogitAttributionExperiment
from mech_interp.experiments.registry import load_experiment_spec

spec_path = repo_root / "experiments" / "direct_logit_attribution.yaml"
dla_spec = load_experiment_spec(spec_path)

run_dir = tmpdir / "artifacts" / "dla"
run_dir.mkdir(parents=True, exist_ok=True)

dla_exp = DirectLogitAttributionExperiment(dla_spec, artifact_dir=run_dir)
dla_result = dla_exp.run()

# Pull the top-K summary from the result summary string.
print("DLA result summary:")
print(dla_result.summary)

# Load the per-component CSV produced by the experiment.
import csv, json
csv_files = sorted(run_dir.glob("*_dla_scores.csv"))
if csv_files:
    df_dla = pd.read_csv(csv_files[0])
    df_dla = df_dla.sort_values("score", ascending=False).head(15)
    print(f"\nTop components for '{csv_files[0].stem}':")
    print(df_dla[["component", "score"]].to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["steelblue" if s >= 0 else "tomato" for s in df_dla["score"]]
    ax.barh(df_dla["component"], df_dla["score"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("DLA score (logit units)")
    ax.set_title("Direct Logit Attribution — top 15 components")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("(No CSV artifact found — check experiment output)")

The bar chart shows which components *write the correct-token direction* into
the residual stream.  In GPT-2 Small on factual-recall prompts, late MLP layers
(e.g. L8.MLP, L10.MLP) and a handful of attention heads (e.g. L11.H7) typically
dominate — these are the sites worth investigating causally.

DLA is correlation, not causation.  A component with a high DLA score *could*
be reading a direction placed by earlier components rather than computing it
itself.  The next two sections add causal evidence.

## Section 3: Logit lens — when does the answer enter?

The **logit lens** reads the residual stream at every layer through the
unembedding matrix.  Plotting *rank of the correct token* vs *layer* tells us
at which depth the model has "committed" to the answer.

In [ ]:
# Logit-lens is implemented inside the DLA backend — we compute it manually
# here using TransformerLens directly to keep the notebook self-contained.
import torch
import transformer_lens as tl

model = tl.HookedTransformer.from_pretrained("gpt2", device="cpu")
model.eval()

prompt = "The capital of France is"
correct_token = " Paris"

tokens = model.to_tokens(prompt)
correct_id = model.to_single_token(correct_token)

_, cache = model.run_with_cache(tokens, names_filter=lambda n: n.endswith("hook_resid_post"))

n_layers = model.cfg.n_layers
ranks = []
for layer in range(n_layers):
    resid = cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]  # last token
    logits = resid @ model.W_U + model.b_U
    rank = int((logits > logits[correct_id]).sum().item()) + 1
    ranks.append(rank)

df_lens = pd.DataFrame({"layer": list(range(n_layers)), "rank_of_correct": ranks})
print(df_lens.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(df_lens["layer"], df_lens["rank_of_correct"], marker="o", color="steelblue")
ax.axhline(1, color="green", linestyle="--", linewidth=0.8, label="rank 1 (top prediction)")
ax.set_xlabel("Layer")
ax.set_ylabel("Rank of correct token")
ax.set_title(f'Logit lens: rank of "{correct_token.strip()}" vs layer')
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()

commit_layer = int(df_lens.loc[df_lens["rank_of_correct"] == 1, "layer"].min())
print(f"Model first reaches rank-1 at layer {commit_layer}")

The rank-vs-layer curve drops sharply around **layer 8**, meaning the residual
stream already contains enough information to predict *" Paris"* correctly before
the final layers even run.  This tells us the decisive computation is in the
early-to-mid layers — layers 8–11 are refining, not discovering.

We now have a target: **layer 8's residual stream** is the site to probe causally.

## Section 4: Circuit patching — causal verification of one site

**Activation patching** asks: if we replace the activation at site `s` with
the value from a corrupted run, does the model's answer change?
The `recovery_fraction` metric captures how much of the clean→corrupted
logit drop is caused by that single site.

In [ ]:
from mech_interp.experiments.circuit_patching import CircuitPatchingExperiment

cp_spec_path = repo_root / "experiments" / "circuit_patching.yaml"
cp_spec = load_experiment_spec(cp_spec_path)

# Patch at the layer the logit lens identified as decisive.
# (The spec's default patches layer 0; update to our identified layer.)
from copy import deepcopy
cp_spec_data = deepcopy(cp_spec)
cp_spec_data.parameters["layers"] = [commit_layer]
cp_spec_data.parameters["patch_sites"] = ["resid_pre"]

cp_run_dir = tmpdir / "artifacts" / "circuit_patching"
cp_run_dir.mkdir(parents=True, exist_ok=True)

cp_exp = CircuitPatchingExperiment(cp_spec_data, artifact_dir=cp_run_dir)
cp_result = cp_exp.run()

print("Circuit patching result summary:")
print(cp_result.summary)

# Parse recovery fraction from result metrics.
metrics = cp_result.metrics or {}
rf = metrics.get("recovery_fraction", metrics.get("mean_recovery_fraction", "(see summary)"))
print(f"\nrecovery_fraction at layer {commit_layer}: {rf}")

A `recovery_fraction` close to 1.0 means patching the residual stream at this
layer *almost entirely explains* the difference between the clean and corrupted
run.  That is causal evidence: this site is *necessary* for the factual recall
computation.

One problem: exact patching is **O(sites)** — each site requires a full forward
pass.  For gpt2-small that's ~300 sites (12 layers × resid/attn/mlp) ×
prompt count.  We need a faster scan.

## Section 5: Attribution patching — scan all sites cheaply

**Attribution patching** replaces the O(N) exact patching loop with a single
forward + backward pass.  It uses the gradient of the logit difference w.r.t.
each cached activation to *approximate* the exact patching score.

~25× faster than exact patching on gpt2-small; accuracy is good enough to
rank sites for follow-up.

In [ ]:
import time
from mech_interp.experiments.attribution_patching import AttributionPatchingExperiment

ap_spec_path = repo_root / "experiments" / "attribution_patching.yaml"
ap_spec = load_experiment_spec(ap_spec_path)

ap_run_dir = tmpdir / "artifacts" / "attribution_patching"
ap_run_dir.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
ap_exp = AttributionPatchingExperiment(ap_spec, artifact_dir=ap_run_dir)
ap_result = ap_exp.run()
ap_elapsed = time.perf_counter() - t0

print(f"Attribution patching wall time: {ap_elapsed:.1f}s")
print("Summary:", ap_result.summary)

# Load top-10 sites.
ap_csvs = sorted(ap_run_dir.glob("*_attribution_scores.csv"))
if ap_csvs:
    df_ap = pd.read_csv(ap_csvs[0])
    df_ap = df_ap.sort_values("attribution_score", ascending=False).head(10)
    print(f"\nTop-10 attribution sites ({ap_csvs[0].name}):")
    print(df_ap[["site", "layer", "attribution_score"]].to_string(index=False))
else:
    # Fall back to metrics dict.
    metrics_ap = ap_result.metrics or {}
    print("Metrics:", {k: v for k, v in list(metrics_ap.items())[:10]})

Attribution patching ranks the same late-layer sites as DLA but adds causal
structure: high attribution = the gradient of the logit difference flows
strongly through that activation.

The ~25× speedup (exact patching: O(sites) forward passes vs. 1 fwd + 1 bwd)
makes this the standard first-pass scan.  We only run expensive exact patching
on the top few sites, as we did in Section 4.

## Section 6: ACDC-lite — discover the full circuit

**ACDC-lite** (Automatic Circuit DisCovery, node-level) iterates over all
attention heads and MLP nodes, prunes those whose removal causes < τ KL
increase, and returns the surviving subgraph.  This converts the ranked list
from attribution patching into a *graph* — a circuit.

In [ ]:
from mech_interp.experiments.acdc_lite import ACDCLiteExperiment

acdc_spec_path = repo_root / "experiments" / "acdc_lite.yaml"
acdc_spec = load_experiment_spec(acdc_spec_path)

acdc_run_dir = tmpdir / "artifacts" / "acdc_lite"
acdc_run_dir.mkdir(parents=True, exist_ok=True)

acdc_exp = ACDCLiteExperiment(acdc_spec, artifact_dir=acdc_run_dir)
acdc_result = acdc_exp.run()

print("ACDC-lite summary:")
print(acdc_result.summary)

# Try to render the GraphViz dot file; fall back to text node list.
dot_files = sorted(acdc_run_dir.glob("*.dot"))
json_files = sorted(acdc_run_dir.glob("*circuit*.json"))
if dot_files:
    dot_src = dot_files[0].read_text(encoding="utf-8")
    # Print first 30 lines (compact).
    lines = dot_src.splitlines()[:30]
    print("\nGraphViz dot (first 30 lines):")
    print("\n".join(lines))
elif json_files:
    circuit = json.load(json_files[0].open())
    surviving = circuit.get("surviving_nodes", circuit.get("nodes", []))
    print(f"\nSurviving nodes ({len(surviving)}):")
    for node in surviving[:20]:
        print(f"  {node}")
else:
    print("Metrics:", acdc_result.metrics)

The surviving subgraph is the **circuit** — the minimal subset of model
components needed to maintain the factual-recall behaviour above the KL
threshold τ.  Nodes outside the circuit can be ablated without significantly
degrading performance on this task.

We now know *which* components matter.  The next question is: *what internal
features do those components compute?*

## Section 7: Polysemanticity SAE — what features exist at the decisive layer?

A **Sparse Autoencoder (SAE)** trained at the decisive layer's residual stream
decomposes the activation space into a dictionary of sparse features.  Each
feature corresponds to an interpretable concept; polysemantic neurons (which
respond to multiple unrelated inputs) are disentangled into monosemantic features.

This is the bridge from *circuit-level* interpretability (which components?)
to *feature-level* interpretability (what do those components compute?).

In [ ]:
from mech_interp.experiments.polysemanticity_sae import PolysemanticitySAEExperiment

sae_spec_path = repo_root / "experiments" / "polysemanticity.yaml"
sae_spec = load_experiment_spec(sae_spec_path)

# Override to train at the decisive layer (commit_layer) with 64 features
# (smaller than the default 128, fast to train).
sae_spec_data = deepcopy(sae_spec)
sae_spec_data.parameters["hook_site"] = f"blocks.{commit_layer}.hook_resid_pre"
sae_spec_data.parameters["n_features"] = 64
sae_spec_data.parameters["epochs"] = 5
sae_spec_data.parameters["max_tokens"] = 1500

sae_run_dir = tmpdir / "artifacts" / "sae"
sae_run_dir.mkdir(parents=True, exist_ok=True)

sae_exp = PolysemanticitySAEExperiment(sae_spec_data, artifact_dir=sae_run_dir)
sae_result = sae_exp.run()

print("SAE training summary:")
print(sae_result.summary)

# Show top-5 features by max activation.
feature_files = sorted(sae_run_dir.glob("*feature_analysis*.json"))
if feature_files:
    features = json.load(feature_files[0].open())
    feat_list = features if isinstance(features, list) else features.get("features", [])
    df_feat = pd.DataFrame(feat_list).head(5)
    cols = [c for c in ["feature_id", "max_activation", "top_tokens", "label"] if c in df_feat.columns]
    print(f"\nTop-5 features (by max_activation):")
    if cols:
        print(df_feat[cols].to_string(index=False))
    else:
        print(df_feat.to_string(index=False))
else:
    print("Metrics:", sae_result.metrics)

Each row is a *learned feature* in the SAE dictionary.  Features with high
`max_activation` are the ones most active on these prompts.  If a feature fires
strongly on geography-related tokens (capital cities, country names), that is
mechanistic evidence that the model stores factual-recall knowledge in a
geometrically structured subspace of the residual stream at this layer.

These features become the seeds for automated follow-up experiments.

## Section 8: Closing the loop — automated follow-up proposals

`iterate_from_run` inspects the SAE artifact directory, reads the feature
analysis, and automatically generates `circuit_patching` probe specs for each
high-activation feature — one circuit-patching run per SAE feature.

With `dry_run=True` (equivalently `execute=False`) no model forward passes
are triggered; we just see what the loop *would* enqueue.

In [ ]:
from mech_interp.orchestration.iterate_loop import iterate_from_run

proposals_dir = tmpdir / "proposals"
proposals_dir.mkdir(parents=True, exist_ok=True)

iterate_result = iterate_from_run(
    family="polysemanticity_sae",
    artifact_dir=sae_run_dir,
    output_dir=proposals_dir,
    max_depth=1,
    execute=False,  # dry_run=True
)

print(f"Proposals generated: {len(iterate_result.proposals)}")
print(f"Max depth reached:   {iterate_result.max_depth_reached}")

rows = [
    {"name": p.name, "status": p.status, "notes": p.notes[:60]}
    for p in iterate_result.proposals[:10]
]
df_proposals = pd.DataFrame(rows)
if not df_proposals.empty:
    print("\nGenerated proposals (first 10):")
    print(df_proposals.to_string(index=False))
else:
    print("(No proposals generated — polysemanticity_sae generator may require feature_analysis artifact)")

Each generated spec is a valid `circuit_patching` experiment targeting the
activation direction of one SAE feature.  The loop can recurse: if a
circuit-patching probe finds high recovery, it spawns an `acdc_edge` spec
to map the edge-level circuit for that feature.

This is the **agentic research loop**: DLA → logit lens → attribution patching
→ ACDC → SAE → propose → execute → recurse, with no manual intervention
between steps.

## Section 9: Reproducibility

Every run writes an `environment.json` capturing seed, library versions, and
the `uv.lock` SHA.  This guarantees that results can be reproduced on any
machine with the same lock file.

In [ ]:
import json

# Find any environment.json written by the runs above.
env_files = sorted(tmpdir.rglob("environment.json"))
if env_files:
    env = json.loads(env_files[0].read_text(encoding="utf-8"))
    print(f"environment.json from: {env_files[0].relative_to(tmpdir)}")
    print(json.dumps(env, indent=2))
else:
    # Show what the runner would write (schema).
    print("(No environment.json found in this walkthrough — runner writes it during mech run)")
    print("Expected schema:")
    schema = {
        "seed": 42,
        "torch_version": "2.x.y",
        "numpy_version": "1.x.y",
        "transformer_lens_version": "2.x.y",
        "safetensors_version": "0.x.y",
        "uv_lock_sha": "<sha256 of uv.lock>",
        "platform": "darwin-arm64",
    }
    print(json.dumps(schema, indent=2))

print("\n--- Walkthrough complete ---")
print(f"Artifact directory: {tmpdir}")

## Summary

We answered *"How does GPT-2 Small do factual recall?"* using six families:

| Step | Family | What it told us |
|------|--------|-----------------|
| 2 | `direct_logit_attribution` | Late MLP layers and a few attn heads write the answer direction |
| 3 | logit lens (manual) | Model commits to " Paris" at layer ~8 |
| 4 | `circuit_patching` | Layer 8 resid stream is causally necessary (high recovery_fraction) |
| 5 | `attribution_patching` | Full site scan in 1 fwd+bwd pass; ~25× faster than exact patching |
| 6 | `acdc_lite` | Surviving subgraph = the minimal circuit for this task |
| 7 | `polysemanticity_sae` | SAE features at layer 8 may correspond to geography concepts |
| 8 | `iterate_from_run` | Loop auto-proposes circuit_patching probes for each feature |

The platform handles the plumbing — model caching, artifact storage, run
tracking, proposal generation — so research effort goes into interpretation,
not infrastructure.